# Caller

Caller is a class provided to enable calling code in different threads and getting the result.

One caller instance is created per thread, and each of those instances can be retrieved by name using the `Caller.get_instance` class method. This method also creates a new thread if one doesn't already exist.

Most methods that perform execution return an async_kernel.Future.

## Usage by the kernel

Kernel uses one of `Caller().call_soon` `Caller.to_thread_by_thread_name` depending on the [header directive](kernel_directive.ipynb#Kernel-directive)

## Example
**This example requires ipywidgets!**

In [ ]:
import random
import threading
import time

import ipywidgets as ipw

outputs = {}


def my_func(n):
    thread = threading.current_thread()
    if not (out := outputs.get(thread)):
        outputs[thread] = out = ipw.HTML(description=thread.name)
        out.style.description_width = "initial"
        display(out)
    sleep_time = random.random() / 4
    out.value = f"started job {n} sleeping {sleep_time * 1000:03.0f} ms"
    time.sleep(sleep_time)
    return n


async def run_forever():
    n = 0
    while True:
        n += 1
        yield Caller.to_thread(my_func, n)


async for fut in Caller.as_completed(run_forever()):
    result = await fut
    print(f"Finished: {result}", end="\r")